# ProofWriter + CLUTRR Neuro-Symbolic Reasoning Datasets

## Overview

This notebook demonstrates loading and processing two peer-reviewed neuro-symbolic reasoning benchmark datasets:

**ProofWriter** (tasksource/proofwriter, ACL 2021): 230,242 examples with natural-language theories (facts + rules), yes/no queries, and gold proof traces. Reasoning depth ranges from 0 to 5+.

**CLUTRR** (kendrivp/CLUTRR_v1_extracted, EMNLP 2019): 95,672 examples with kinship narrative stories, relational queries, and explicit proof_state chains. Covers multi-hop kinship inference.

Both datasets are standardized to a unified schema for neuro-symbolic pipeline evaluation, with gold reasoning traces for auditing logical deduction paths.

## Setup: Install Dependencies

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is NOT pre-installed on Colab, always install
_pip('loguru==0.7.3')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

## Imports

In [ ]:
import json
import re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from loguru import logger
import sys

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

## Data Loading: GitHub URL Pattern with Local Fallback

This cell implements the data loading pattern for Colab compatibility. It tries to fetch `mini_demo_data.json` from GitHub first, then falls back to the local file if the URL is not yet available.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7fd9e8-typed-unification-failure-recovery-a-str/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub or local fallback."""
    try:
        import urllib.request
        logger.info(f"Attempting to load from GitHub: {GITHUB_DATA_URL}")
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=5) as response:
            data = json.loads(response.read().decode())
            logger.info("Successfully loaded data from GitHub")
            return data
    except Exception as e:
        logger.warning(f"GitHub load failed ({type(e).__name__}), trying local file")
    
    if Path("mini_demo_data.json").exists():
        logger.info("Loading from local mini_demo_data.json")
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local disk")

## Load Demo Data

In [ ]:
data = load_data()
logger.info(f"Loaded data with {len(data['datasets'])} datasets")

## Configuration: Demo Parameters

These parameters control demo scope. Set to minimal values for quick prototyping:
- `max_examples_per_dataset`: max examples to process from each dataset (None = all)
- `truncate_field_length`: max characters to display for long fields like proofs

In [ ]:
# Demo config parameters
max_examples_per_dataset = None  # None = all examples in mini_demo_data.json
truncate_field_length = 150  # Truncate long fields for display

## Processing: Extract and Standardize Examples

This section processes examples from the standardized schema. Each example has:
- `input`: combined natural language context/story and query
- `output`: expected answer (true/false for ProofWriter, kinship term for CLUTRR)
- `metadata_*`: reasoning depth, proof traces, and other diagnostic fields

In [ ]:
# Extract all examples from datasets
all_examples = []
dataset_stats = {}

for dataset_info in data['datasets']:
    dataset_name = dataset_info['dataset']
    examples = dataset_info['examples']
    
    # Limit examples if configured
    if max_examples_per_dataset:
        examples = examples[:max_examples_per_dataset]
    
    # Record statistics
    dataset_stats[dataset_name] = {
        'num_examples': len(examples),
        'splits': {},
        'reasoning_depths': {}
    }
    
    # Aggregate by split and reasoning depth
    for example in examples:
        split = example.get('metadata_split', 'unknown')
        depth = example.get('metadata_reasoning_depth', 0)
        
        dataset_stats[dataset_name]['splits'][split] = dataset_stats[dataset_name]['splits'].get(split, 0) + 1
        dataset_stats[dataset_name]['reasoning_depths'][depth] = dataset_stats[dataset_name]['reasoning_depths'].get(depth, 0) + 1
    
    all_examples.extend([(dataset_name, ex) for ex in examples])
    logger.info(f"Processed {dataset_name}: {len(examples)} examples")

logger.info(f"Total examples processed: {len(all_examples)}")

## ProofWriter Analysis

Examine ProofWriter examples: propositional logic reasoning with fact/rule counts and proof depth.

In [ ]:
proofwriter_examples = [(ds, ex) for ds, ex in all_examples if ds == 'proofwriter']

if proofwriter_examples:
    logger.info(f"\nProofWriter examples ({len(proofwriter_examples)}):")
    for ds, ex in proofwriter_examples[:2]:  # Show first 2
        logger.info(f"  Input: {ex['input'][:100]}...")
        logger.info(f"  Output: {ex['output']}")
        logger.info(f"  Depth: {ex.get('metadata_reasoning_depth', 0)}, "
                    f"Facts: {ex.get('metadata_num_facts', 0)}, "
                    f"Rules: {ex.get('metadata_num_rules', 0)}")
else:
    logger.info("No ProofWriter examples in this dataset")

## CLUTRR Analysis

Examine CLUTRR examples: kinship reasoning with multi-hop narrative stories and proof states.

In [ ]:
clutrr_examples = [(ds, ex) for ds, ex in all_examples if ds == 'clutrr']

if clutrr_examples:
    logger.info(f"\nCLUTRR examples ({len(clutrr_examples)}):")
    for ds, ex in clutrr_examples[:2]:  # Show first 2
        logger.info(f"  Input: {ex['input'][:100]}...")
        logger.info(f"  Output: {ex['output']}")
        logger.info(f"  Depth: {ex.get('metadata_reasoning_depth', 0)}, "
                    f"F_comb: {ex.get('metadata_f_comb', 'N/A')}")
else:
    logger.info("No CLUTRR examples in this dataset")

## Results Summary and Visualization

Aggregate statistics across datasets and visualize reasoning depth distributions.

In [ ]:
# Build summary table
summary_data = []
for dataset_name, stats in dataset_stats.items():
    row = {
        'Dataset': dataset_name,
        'Total Examples': stats['num_examples'],
        'Train': stats['splits'].get('train', 0),
        'Test': stats['splits'].get('test', 0),
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print("\n=== Dataset Summary ===")
print(summary_df.to_string(index=False))
print(f"\nTotal across datasets: {len(all_examples)} examples")

## Reasoning Depth Distribution

In [ ]:
# Extract reasoning depth for visualization
depth_data = {'proofwriter': {}, 'clutrr': {}}

for ds, ex in all_examples:
    depth = ex.get('metadata_reasoning_depth', 0)
    if ds not in depth_data:
        depth_data[ds] = {}
    depth_data[ds][depth] = depth_data[ds].get(depth, 0) + 1

# Print depth distributions
print("\n=== Reasoning Depth Distribution ===")
for ds_name, depths in depth_data.items():
    if depths:
        print(f"\n{ds_name}:")
        for depth in sorted(depths.keys()):
            count = depths[depth]
            print(f"  Depth {depth}: {count} examples")

## Visualization: Reasoning Complexity by Dataset

In [ ]:
# Plot reasoning depth distribution for each dataset
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for idx, (ds_name, depths) in enumerate(depth_data.items()):
    if depths:
        ax = axes[idx]
        d_keys = sorted(depths.keys())
        d_vals = [depths[k] for k in d_keys]
        
        ax.bar([str(d) for d in d_keys], d_vals, color='steelblue', alpha=0.7)
        ax.set_xlabel('Reasoning Depth')
        ax.set_ylabel('Number of Examples')
        ax.set_title(f'{ds_name.upper()} - Reasoning Depth Distribution')
        ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

logger.info("Visualization complete")

## Example Inspection

Show a sample example from each dataset with full field details.

In [ ]:
# Show detailed example from each dataset
for ds_name in ['proofwriter', 'clutrr']:
    examples = [ex for d, ex in all_examples if d == ds_name]
    if examples:
        ex = examples[0]
        print(f"\n{'='*60}")
        print(f"Sample {ds_name.upper()} Example:")
        print(f"{'='*60}")
        print(f"Input (truncated): {ex['input'][:200]}...")
        print(f"Output: {ex['output']}")
        print(f"\nMetadata:")
        for key, value in ex.items():
            if key.startswith('metadata_'):
                # Truncate long values
                if isinstance(value, str) and len(value) > truncate_field_length:
                    value = value[:truncate_field_length] + '...'
                print(f"  {key}: {value}")

## Schema Validation Summary

Verify that all examples conform to the expected schema (exp_sel_data_out format).

In [ ]:
# Validate schema compliance
required_fields = {'input', 'output'}
valid_count = 0
invalid_count = 0

for ds, ex in all_examples:
    if all(field in ex for field in required_fields):
        valid_count += 1
    else:
        invalid_count += 1
        logger.warning(f"Invalid example in {ds}: missing required fields")

print(f"\n=== Schema Validation ===")
print(f"Valid examples: {valid_count}")
print(f"Invalid examples: {invalid_count}")
print(f"Validation rate: {100 * valid_count / len(all_examples):.1f}%")

if invalid_count == 0:
    logger.info("✓ All examples conform to exp_sel_data_out schema")
else:
    logger.warning(f"✗ {invalid_count} examples do not conform to schema")